In [8]:
import pandas as pd
from pathlib import Path
import time

file_path = Path("data/Online Retail.xlsx")

print("File exists:", file_path.exists())
if file_path.exists():
    print("File size (MB):", round(file_path.stat().st_size / (1024 * 1024), 2))

start = time.time()

df = pd.read_excel(
    file_path,
    engine="openpyxl"
)

print("Loaded successfully!")
print("Shape:", df.shape)
print("Time taken:", round(time.time() - start, 2), "seconds")

display(df.head())

File exists: True
File size (MB): 22.62
Loaded successfully!
Shape: (541909, 8)
Time taken: 109.47 seconds


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
file_path = Path("data/Online Retail.xlsx")
csv_path = Path("data/online_retail.csv")

start = time.time()
print("Reading Excel file... Please wait.")

df = pd.read_excel(file_path, engine="openpyxl")

print("Excel loaded:", df.shape)
df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)
print("Time taken:", round(time.time() - start, 2), "seconds")

Reading Excel file... Please wait.
Excel loaded: (541909, 8)
CSV saved: data\online_retail.csv
Time taken: 100.3 seconds


In [7]:
df = pd.read_csv("data/online_retail.csv")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (541909, 8)

Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Data types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


In [ ]:

df = pd.read_csv("data/online_retail.csv")

# Convert data types
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = df["CustomerID"].astype("Int64")

# Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()

# Remove records missing essential customer/product/date information
df_clean = df_clean.dropna(
    subset=["CustomerID", "Description", "InvoiceDate"]
).copy()

# Identify cancellations and returns
df_clean["IsCancelled"] = (
    df_clean["InvoiceNo"].astype(str).str.startswith("C")
)

df_clean["IsReturnOrInvalidQty"] = df_clean["Quantity"] <= 0

# Calculate line amount for analysis
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

# Save cleaned audit dataset
output_path = Path("data/online_retail_clean.csv")
df_clean.to_csv(output_path, index=False)

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print("Cancelled invoice rows:", df_clean["IsCancelled"].sum())
print("Rows with quantity <= 0:", df_clean["IsReturnOrInvalidQty"].sum())
print("Cleaned file saved to:", output_path)

Original rows: 541909
Cleaned rows: 401604
Rows removed: 140305
Cancelled invoice rows: 8872
Rows with quantity <= 0: 8872
Cleaned file saved to: data\online_retail_clean.csv


In [ ]:
# Load our cleaned dataset
df_clean = pd.read_csv(
    "data/online_retail_clean.csv",
    parse_dates=["InvoiceDate"]
)

# Keep only valid purchase lines:
# - Not a cancelled invoice
# - Positive quantity
# - Positive unit price
sales = df_clean[
    (~df_clean["IsCancelled"].astype(bool))
    & (df_clean["Quantity"] > 0)
    & (df_clean["UnitPrice"] > 0)
].copy()

# Calculate revenue for each transaction line
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]

# Save the sales dataset
sales.to_csv("data/online_retail_sales.csv", index=False)

print("Sales rows:", len(sales))
print("Unique customers:", sales["CustomerID"].nunique())
print("Unique invoices:", sales["InvoiceNo"].nunique())
print("Total revenue: £{:,.2f}".format(sales["Revenue"].sum()))
print("Date range:", sales["InvoiceDate"].min(), "to", sales["InvoiceDate"].max())
print("\nSales dataset preview:")
display(sales.head())

Sales rows: 392692
Unique customers: 4338
Unique invoices: 18532
Total revenue: £8,887,208.89
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00

Sales dataset preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,IsReturnOrInvalidQty,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,20.34
